# Homework 5

### Homework Initial Conditions

#### Given

##### Cartesian State (ECI Position & Velocity)

| Component | Value              | Units     |
|-----------|--------------------|-----------|
| X         | 78000000.00000000  | meters    |
| Y         | 0.00000000         | meters    |
| Z         | 0.00000000         | meters    |
| XD        | 0.00000000         | m/sec     |
| YD        | 6190.8777608       | m/sec     |
| ZD        | 3574.304854        | m/sec     |

##### Classical Orbital Elements

| Element | Value          | Units |
|---------|----------------|-------|
| a       | 78000000.00000000 | meters |
| e       | 0              | —     |
| i       | 30             | deg   |
| Ω       | 0              | deg   |
| ωp      | 0              | deg   |
| ν       | 0              | deg   |

**Notes:**
- Circular orbit (e = 0)
- Semi-major axis = 78,000 km (very high orbit)
- Inclination = 30°
- All angles are zero → equatorial plane reference with argument of perigee and true anomaly aligned

##### Integration Parameters
- **Step size**: 60 seconds

## Problem 1

Use **RK4** to integrate the vector **10 steps** using

$
\vec{a} = -\frac{\mu}{r^3} \vec{r}
$

### Tasks

- Implement classical 4th-order Runge-Kutta (RK4) integration
- Perform **10 integration steps**

### Notes 
#### Classic Runge–Kutta 4th Order (RK4)

One of the simplest and most commonly used Runge–Kutta methods.

##### Update formula

$
\vec{y}(t + h) = \vec{y}_0 + h \left( \frac{\vec{k}_1}{6} + \frac{\vec{k}_2}{3} + \frac{\vec{k}_3}{3} + \frac{\vec{k}_4}{6} \right)
$

##### The four increments (stages)

$
\begin{aligned}
\vec{k}_1 &= \vec{f}(t_0, \vec{y}_0) \\[1.2em]
\vec{k}_2 &= \vec{f}\left(t_0 + \frac{h}{2},\ \vec{y}_0 + \frac{h}{2} \vec{k}_1 \right) \\[1.2em]
\vec{k}_3 &= \vec{f}\left(t_0 + \frac{h}{2},\ \vec{y}_0 + \frac{h}{2} \vec{k}_2 \right) \\[1.2em]
\vec{k}_4 &= \vec{f}\left(t_0 + h,\ \vec{y}_0 + h \vec{k}_3 \right)
\end{aligned}
$

##### General Form
$
\vec{y} = \begin{bmatrix} \vec{r} \\ \dot{\vec{r}} \end{bmatrix}
\quad \longrightarrow \quad \text{6×1 state vector}
$

$
\vec{f} = \begin{bmatrix} \dot{\vec{r}} \\ \vec{a}(\vec{r}, \dot{\vec{r}}, t) \end{bmatrix}
$

##### Two-Body Example (Keplerian / central force)

$
\vec{y} = \begin{bmatrix} \vec{r} \\ \dot{\vec{r}} \end{bmatrix}
\quad \longrightarrow \quad \text{6×1 state vector}
$

$
\vec{f} = \begin{bmatrix} \dot{\vec{r}} \\[1em] -\dfrac{\mu}{r^3} \vec{r} \end{bmatrix}
$

In [7]:
from standards import *
import math
import copy

# initial conditions:
pos_ = Vector3(7800000.00000000, 0.00000000, 0.00000000) # m
vel_ = Vector3(0.00000000, 6190.8777608, 3574.304854) # m/s

mu = 3.986004418e14
def compute_rk_k1(mu:float, pos_: Vector3, vel_: Vector3) -> Vector6:
    f_pos = vel_
    f_vel = -(mu/(math.pow(pos_.magnitude(), 3)))*pos_
    return Vector6(f_pos.x, f_pos.y, f_pos.z, f_vel.x, f_vel.y, f_vel.z)

def compute_rk_k2(mu:float,h: float, y_0: Vector6, k1: Vector6) -> Vector6:
    y = y_0 + h*k1/2
    y_top = Vector3(y.x, y.y, y.z)
    y_bottom = Vector3(y.xd, y.yd, y.zd)
    f_top = y_bottom
    f_bottom = -(mu/(math.pow(y_top.magnitude(), 3)))*y_top
    return Vector6(f_top.x, f_top.y, f_top.z, f_bottom.x, f_bottom.y, f_bottom.z)

def compute_rk_k3(mu:float,h: float, y_0: Vector6, k2: Vector6) -> Vector6:
    y = y_0 + h*k2/2
    y_top = Vector3(y.x, y.y, y.z)
    y_bottom = Vector3(y.xd, y.yd, y.zd)
    f_top = y_bottom
    f_bottom = -(mu/(math.pow(y_top.magnitude(), 3)))*y_top
    return Vector6(f_top.x, f_top.y, f_top.z, f_bottom.x, f_bottom.y, f_bottom.z)

def compute_rk_k4(mu:float,h: float, y_0: Vector6, k3: Vector6) -> Vector6:
    y = y_0 + h*k3
    y_top = Vector3(y.x, y.y, y.z)
    y_bottom = Vector3(y.xd, y.yd, y.zd)
    f_top = y_bottom
    f_bottom = -(mu/(math.pow(y_top.magnitude(), 3)))*y_top
    return Vector6(f_top.x, f_top.y, f_top.z, f_bottom.x, f_bottom.y, f_bottom.z)

def compute_rk_2_body_step(mu: float, pos_:Vector3, vel_:Vector3, h: float, y_0:Vector6) -> Vector6:
    k1 = compute_rk_k1(mu, pos_, vel_)
    k2 = compute_rk_k2(mu, h, y_0, k1)
    k3 = compute_rk_k3(mu, h, y_0, k2)
    k4 = compute_rk_k4(mu, h, y_0, k3)
    step = y_0 + h*(k1/6+k2/3+k3/3+k4/6)
    return step

# compute 10 steps:
y_0 = Vector6(pos_.x, pos_.y, pos_.z, vel_.x, vel_.y, vel_.z)
h = 60
class Step:
    T: float
    RK_X: float
    RK_Y: float
    RK_Z: float
    RK_XD: float
    RK_YD: float
    RK_ZD: float

    def __init__(self, T, RK_X, RK_Y, RK_Z, RK_XD, RK_YD, RK_ZD):
        self.T = T
        self.RK_X = RK_X
        self.RK_Y = RK_Y
        self.RK_Z = RK_Z
        self.RK_XD = RK_XD
        self.RK_YD = RK_YD
        self.RK_ZD = RK_ZD

## F and G calculations:
###################################################################################################
keps = KeplerianElements(pos_, vel_)
keps.a = 7800000
keps.ecc = 0
keps.inc = math.radians(30)
keps.argp = 0
keps.raan = 0
keps.ta = 0
E_0 = compute_eccentric_anomaly(keps.ta, keps.ecc)
f_g_steps = []
f_g_steps.append(Step(0, y_0.x, y_0.y, y_0.z, y_0.xd, y_0.yd, y_0.zd))

for i in range(10):
    delta_t = h*(i+1)
    E_propagated, _, nu_propagated = compute_propagate_nu_given_delta_t(keps, delta_t)
    f, g, f_dot, g_dot = compute_f_g_f_dot_g_dot_no_final_perifocal(keps.ecc, keps.ta, nu_propagated, keps.a, keps.mu_earth, (E_propagated - E_0), delta_t)
    r_ = f*pos_ + g*vel_
    r_dot_ = f_dot*pos_ + g_dot*vel_
    step = Step(0, r_.x, r_.y, r_.z, r_dot_.x, r_dot_.y, r_dot_.z)
    f_g_steps.append(step)
###################################################################################################


steps = []
steps.append(Step(0, y_0.x, y_0.y, y_0.z, y_0.xd, y_0.yd, y_0.zd))
temp_pos = copy.deepcopy(pos_)
temp_vel = copy.deepcopy(vel_)
for i in range(10):
    t = h*(i+1)
    step = compute_rk_2_body_step(mu, temp_pos, temp_vel, h, y_0)
    steps.append(Step(t, step.x, step.y, step.z, step.xd, step.yd, step.zd))
    y_0 = step
    temp_pos = Vector3(step.x, step.y, step.z)
    temp_vel = Vector3(step.xd, step.yd, step.zd)

### Check your work

Compare your RK4 results to results obtained using the **f and g** functions  
(should get approximately the **same results**)

Hints from class:

- Use the **ΔE approach** as shown in Example 1
- Recall from the exam that:

$
\vec{r}_{\text{ECI}} = f \vec{r}_{o,\text{ECI}} + g \dot{\vec{r}}_{o,\text{ECI}}
$

In [8]:
from IPython.display import Markdown, display

header = "| Time (s) | FG_X (m) | FG_Y (m) | FG_Z (m) | RK_X (m) | RK_Y (m) | RK_Z (m) | pos diff (m) | vel diff (m)|"
separator = "|----------|-------|-------|-------|-------|-------|-------|-------|-------|"

rows = []
for i in range(11):
    row = f"| {steps[i].T:7.1f} | {f_g_steps[i].RK_X:11.3f} | {f_g_steps[i].RK_Y:11.3f} | {f_g_steps[i].RK_Z:11.3f} | {steps[i].RK_X:11.3f} | {steps[i].RK_Y:11.3f} | {steps[i].RK_Z:11.3f} | \
        {abs(Vector3(f_g_steps[i].RK_X, f_g_steps[i].RK_Y, f_g_steps[i].RK_Z).magnitude() - Vector3(steps[i].RK_X, steps[i].RK_Y, steps[i].RK_Z).magnitude()):11.3f} | \
        {abs(Vector3(f_g_steps[i].RK_XD, f_g_steps[i].RK_YD, f_g_steps[i].RK_ZD).magnitude() - Vector3(steps[i].RK_XD, steps[i].RK_YD, steps[i].RK_ZD).magnitude()):11.7f}"
    rows.append(row)

table_md = "\n".join([header, separator] + rows)
display(Markdown(f"### RK4 Integration Steps\n\n{table_md}"))

### RK4 Integration Steps

| Time (s) | FG_X (m) | FG_Y (m) | FG_Z (m) | RK_X (m) | RK_Y (m) | RK_Z (m) | pos diff (m) | vel diff (m)|
|----------|-------|-------|-------|-------|-------|-------|-------|-------|
|     0.0 | 7800000.000 |       0.000 |       0.000 | 7800000.000 |       0.000 |       0.000 |               0.000 |           0.0000000
|    60.0 | 7788210.059 |  371265.493 |  214350.227 | 7788210.057 |  371265.464 |  214350.210 |               0.003 |           0.0000002
|   120.0 | 7752875.877 |  741408.627 |  428052.460 | 7752875.875 |  741408.570 |  428052.427 |               0.010 |           0.0000034
|   180.0 | 7694104.272 | 1109310.437 |  640460.664 | 7694104.269 | 1109310.352 |  640460.614 |               0.020 |           0.0000098
|   240.0 | 7612072.915 | 1473858.733 |  850932.715 | 7612072.910 | 1473858.619 |  850932.650 |               0.033 |           0.0000191
|   300.0 | 7507029.790 | 1833951.463 | 1058832.345 | 7507029.785 | 1833951.321 | 1058832.263 |               0.049 |           0.0000315
|   360.0 | 7379292.450 | 2188500.044 | 1263531.059 | 7379292.444 | 2188499.875 | 1263530.961 |               0.069 |           0.0000468
|   420.0 | 7229247.052 | 2536432.655 | 1464410.040 | 7229247.045 | 2536432.457 | 1464409.926 |               0.092 |           0.0000650
|   480.0 | 7057347.194 | 2876697.472 | 1660862.019 | 7057347.186 | 2876697.248 | 1660861.890 |               0.118 |           0.0000860
|   540.0 | 6864112.539 | 3208265.855 | 1852293.110 | 6864112.529 | 3208265.604 | 1852292.965 |               0.147 |           0.0001096
|   600.0 | 6650127.247 | 3530135.453 | 2038124.604 | 6650127.235 | 3530135.175 | 2038124.443 |               0.178 |           0.0001358

**Redo** the RK4 integration using **units of Earth Radii (ER)** and **hours**
  - Don't forget to convert/change the units of both **r** and **μ** consistently

In [9]:
ER = 6378137 #m
# MU is in m^3 / s^2
mu = mu * (60*60)**2 * (1/(ER**3))
pos_ = pos_/ER
vel_ = (vel_/ER)*(60*60)
h = h/(60*60)
y_0 = Vector6(pos_.x, pos_.y, pos_.z, vel_.x, vel_.y, vel_.z)

steps = []
steps.append(Step(0, y_0.x, y_0.y, y_0.z, y_0.xd, y_0.yd, y_0.zd))
temp_pos = copy.deepcopy(pos_)
temp_vel = copy.deepcopy(vel_)
for i in range(10):
    t = h*(i+1)
    step = compute_rk_2_body_step(mu, temp_pos, temp_vel, h, y_0)
    steps.append(Step(t, step.x, step.y, step.z, step.xd, step.yd, step.zd))
    y_0 = step
    temp_pos = Vector3(step.x, step.y, step.z)
    temp_vel = Vector3(step.xd, step.yd, step.zd)

header = "| Time (s) | RK_X (ER) | RK_Y (ER) | RK_Z (ER) | RK_XD (ER/hr) | RK_YD (ER/hr) | RK_ZD (ER/hr)|"
separator = "|----------|-------|-------|-------|-------|-------|-------|"

rows = []
for i in range(11):
    row = f"| {steps[i].T:7.1f} | {steps[i].RK_X:11.6f} | {steps[i].RK_Y:11.6f} | {steps[i].RK_Z:11.6f} | {steps[i].RK_XD:11.6f} | {steps[i].RK_YD:11.6f} | {steps[i].RK_ZD:11.6f} |"
    rows.append(row)

table_md = "\n".join([header, separator] + rows)
display(Markdown(f"### RK4 Integration Steps\n\n{table_md}"))

### RK4 Integration Steps

| Time (s) | RK_X (ER) | RK_Y (ER) | RK_Z (ER) | RK_XD (ER/hr) | RK_YD (ER/hr) | RK_ZD (ER/hr)|
|----------|-------|-------|-------|-------|-------|-------|
|     0.0 |    1.222928 |    0.000000 |    0.000000 |    0.000000 |    3.494306 |    2.017438 |
|     0.0 |    1.221079 |    0.058209 |    0.033607 |   -0.221763 |    3.489024 |    2.014389 |
|     0.0 |    1.215539 |    0.116242 |    0.067112 |   -0.442856 |    3.473195 |    2.005250 |
|     0.1 |    1.206325 |    0.173924 |    0.100415 |   -0.662610 |    3.446866 |    1.990049 |
|     0.1 |    1.193463 |    0.231080 |    0.133414 |   -0.880361 |    3.410117 |    1.968832 |
|     0.1 |    1.176994 |    0.287537 |    0.166010 |   -1.095451 |    3.363059 |    1.941663 |
|     0.1 |    1.156967 |    0.343125 |    0.198103 |   -1.307229 |    3.305834 |    1.908624 |
|     0.1 |    1.133442 |    0.397676 |    0.229598 |   -1.515055 |    3.238615 |    1.869815 |
|     0.1 |    1.106490 |    0.451025 |    0.260399 |   -1.718301 |    3.161606 |    1.825354 |
|     0.1 |    1.076194 |    0.503010 |    0.290413 |   -1.916352 |    3.075039 |    1.775375 |
|     0.2 |    1.042644 |    0.553474 |    0.319549 |   -2.108611 |    2.979177 |    1.720028 |